In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Update these paths if your files are stored elsewhere.
CITY_DAY_PATH = Path('/content/city_day.csv')
AQI_PATH = Path('/content/aqi.csv')
OUTPUT_PATH = Path('/content/master_aqi_pipeline.csv')

if not CITY_DAY_PATH.exists() or not AQI_PATH.exists():
    raise FileNotFoundError(
        "Could not find city_day.csv and/or aqi.csv. Upload both files to /content "
        "or change CITY_DAY_PATH and AQI_PATH to the correct locations."
    )

df_2015 = pd.read_csv(CITY_DAY_PATH)
df_2023 = pd.read_csv(AQI_PATH)

df_2023 = df_2023.rename(columns={
    'area': 'City', 'date': 'Date', 'aqi_value': 'AQI',
    'air_quality_status': 'AQI_Bucket'
})

required = {'City', 'Date', 'AQI'}
for name, frame in [('city_day.csv', df_2015), ('aqi.csv', df_2023)]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

for frame in (df_2015, df_2023):
    frame['City'] = frame['City'].astype('string').str.strip().str.title()
    frame['Date'] = pd.to_datetime(frame['Date'], errors='coerce', dayfirst=True)
    frame['AQI'] = pd.to_numeric(frame['AQI'], errors='coerce')

master_df = pd.concat([df_2015, df_2023], ignore_index=True, sort=False)
master_df = master_df.dropna(subset=['City', 'Date', 'AQI'])
master_df = master_df[master_df['City'].str.len() > 0]
master_df = master_df.drop_duplicates(subset=['City', 'Date'], keep='last')
master_df = master_df.sort_values(['City', 'Date']).reset_index(drop=True)
master_df.to_csv(OUTPUT_PATH, index=False)

print(f"Loaded {len(master_df):,} city-date records across {master_df['City'].nunique()} cities.")
print('Cities available:', ', '.join(sorted(master_df['City'].unique())))

Loaded 244,745 city-date records across 291 cities.
Cities available: Agartala, Agra, Ahmedabad, Ahmednagar, Aizawl, Ajmer, Akola, Alwar, Amaravati, Ambala, Amravati, Amritsar, Anantapur, Angul, Ankleshwar, Araria, Ariyalur, Arrah, Asansol, Aurangabad, Baddi, Badlapur, Bagalkot, Baghpat, Bahadurgarh, Balasore, Ballabgarh, Banswara, Baran, Barbil, Bareilly, Baripada, Barmer, Barrackpore, Bathinda, Begusarai, Belapur, Belgaum, Bengaluru, Bettiah, Bhagalpur, Bharatpur, Bhilai, Bhilwara, Bhiwadi, Bhiwandi, Bhiwani, Bhopal, Bhubaneswar, Bidar, Bihar Sharif, Bikaner, Bilaspur, Bileipada, Boisar, Brajrajnagar, Bulandshahr, Bundi, Buxar, Byasanagar, Byrnihat, Chamarajanagar, Chandigarh, Chandrapur, Charkhi Dadri, Chengalpattu, Chennai, Chhal, Chhapra, Chikkaballapur, Chikkamagaluru, Chittoor, Chittorgarh, Churu, Coimbatore, Cuddalore, Cuttack, Damoh, Darbhanga, Dausa, Davanagere, Dehradun, Delhi, Dewas, Dhanbad, Dharuhera, Dharwad, Dholpur, Dhule, Dindigul, Dungarpur, Durgapur, Eloor, Ernakula

In [3]:
# Create calendar features for every city (no city is hard-coded).
df = pd.read_csv(OUTPUT_PATH, parse_dates=['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfYear'] = df['Date'].dt.dayofyear
df['Weekday'] = df['Date'].dt.dayofweek
df['Season'] = df['Month'].map({12:'Winter', 1:'Winter', 2:'Spring', 3:'Spring',
                                4:'Summer', 5:'Summer', 6:'Summer', 7:'Summer',
                                8:'Monsoon', 9:'Monsoon', 10:'Autumn', 11:'Autumn'})
df['Is_Lohri_Month'] = (df['Month'] == 1).astype(int)
df['Is_Diwali_Season'] = df['Month'].isin([10, 11]).astype(int)
df['Is_Paddy_Harvesting'] = (
    df['Month'].isin([10, 11]) | ((df['Month'] == 9) & (df['Day'] >= 15))
).astype(int)

# Keep only records with target AQI and valid city/date.
df = df.dropna(subset=['City', 'Date', 'AQI']).copy()
print('Rows:', len(df), '| Cities:', df['City'].nunique())
display(df[['City', 'Date', 'AQI', 'Year', 'Month', 'Day', 'Season']].head())

Rows: 244745 | Cities: 291


,City,Date,AQI,Year,Month,Day,Season
0,Agartala,2022-04-01,78.0,2022,4,1,Summer
1,Agartala,2022-04-02,77.0,2022,4,2,Summer
2,Agartala,2022-04-03,76.0,2022,4,3,Summer
3,Agartala,2022-04-04,57.0,2022,4,4,Summer
4,Agartala,2022-04-05,64.0,2022,4,5,Summer


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

FEATURES = ['City', 'Year', 'Month', 'Day', 'DayOfYear', 'Weekday',
            'Is_Lohri_Month', 'Is_Diwali_Season', 'Is_Paddy_Harvesting']
TARGET = 'AQI'

model_df = df[FEATURES + [TARGET]].copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[TARGET, 'City'])
X = model_df[FEATURES]
y = model_df[TARGET]

print('Rows available for modelling:', len(model_df))
print('Features:', FEATURES)


Rows available for modelling: 244745
Features: ['City', 'Year', 'Month', 'Day', 'DayOfYear', 'Weekday', 'Is_Lohri_Month', 'Is_Diwali_Season', 'Is_Paddy_Harvesting']


In [5]:
# Split data and configure a faster model.
# Fewer trees and bounded depth reduce training time.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(transformers=[
    ('city', OneHotEncoder(handle_unknown='ignore'), ['City']),
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]),
     [c for c in FEATURES if c != 'City'])
])

MODEL_PATH = '/content/city_aqi_model_fast.joblib'
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=80,       # was 300
        max_depth=14,          # limits tree complexity
        min_samples_leaf=3,    # less overfitting and smaller trees
        max_samples=0.8,       # each tree uses 80% of training rows
        random_state=42,
        n_jobs=-1
    ))
])

print('Training rows:', len(X_train), '| Test rows:', len(X_test))
print('Faster model pipeline is ready.')


Training rows: 195796 | Test rows: 48949
Faster model pipeline is ready.


In [6]:
import os

# If a compatible model was already saved during this runtime, load it.
# Otherwise train and save it.
if os.path.exists(MODEL_PATH):
    try:
        saved = joblib.load(MODEL_PATH)
        if saved.get('features') == FEATURES:
            model = saved['model']
            print('Loaded saved model; training skipped.')
        else:
            raise ValueError('Saved model feature list differs.')
    except Exception:
        print('Saved model could not be reused. Training a new model...')
        model.fit(X_train, y_train)
        joblib.dump({'model': model, 'features': FEATURES,
                     'cities': sorted(df['City'].dropna().unique())}, MODEL_PATH)
        print('Training complete and model saved.')
else:
    print('Training faster Random Forest model...')
    model.fit(X_train, y_train)
    joblib.dump({'model': model, 'features': FEATURES,
                 'cities': sorted(df['City'].dropna().unique())}, MODEL_PATH)
    print('Training complete and model saved.')


Training faster Random Forest model...
Training complete and model saved.


In [7]:
# Evaluate and save (the model is also saved in the training cell).
pred = model.predict(X_test)
print('MAE:', round(mean_absolute_error(y_test, pred), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, pred)), 2))
print('R²:', round(r2_score(y_test, pred), 3))
joblib.dump({'model': model, 'features': FEATURES,
             'cities': sorted(df['City'].dropna().unique())}, MODEL_PATH)
print('Saved model to', MODEL_PATH)


MAE: 43.43
RMSE: 59.15
R²: 0.398
Saved model to /content/city_aqi_model_fast.joblib


In [8]:
# USER INPUT: select a city and date, then predict AQI.
from datetime import date

available_cities = sorted(df['City'].dropna().unique().tolist())

def make_prediction(city, prediction_date):
    city = str(city).strip().title()
    prediction_date = pd.to_datetime(prediction_date, errors='raise')
    if city not in available_cities:
        raise ValueError(f"'{city}' is not in the training data. Choose one of: {available_cities}")

    month = prediction_date.month
    row = pd.DataFrame([{
        'City': city,
        'Year': prediction_date.year,
        'Month': month,
        'Day': prediction_date.day,
        'DayOfYear': prediction_date.dayofyear,
        'Weekday': prediction_date.dayofweek,
        'Is_Lohri_Month': int(month == 1),
        'Is_Diwali_Season': int(month in [10, 11]),
        'Is_Paddy_Harvesting': int(month in [10, 11] or (month == 9 and prediction_date.day >= 15))
    }], columns=FEATURES)
    estimate = float(model.predict(row)[0])
    return max(0, estimate)

# Preferred interactive dropdown in Jupyter/Colab; falls back to text input if
# ipywidgets is unavailable.
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    city_dropdown = widgets.Dropdown(options=available_cities, description='City:',
                                     value='Chandigarh' if 'Chandigarh' in available_cities else available_cities[0],
                                     layout=widgets.Layout(width='350px'))
    date_picker = widgets.DatePicker(description='Prediction date:', value=date.today())
    predict_button = widgets.Button(description='Predict AQI', button_style='primary')
    output = widgets.Output()

    def on_predict_clicked(_):
        with output:
            clear_output()
            if date_picker.value is None:
                print('Please select a date.')
                return
            estimate = make_prediction(city_dropdown.value, date_picker.value)
            print(f'City: {city_dropdown.value}')
            print(f'Date: {date_picker.value}')
            print(f'Predicted AQI: {estimate:.1f}')
            print('Note: this is a city/date-based estimate, not a live AQI reading.')

    predict_button.on_click(on_predict_clicked)
    display(widgets.VBox([city_dropdown, date_picker, predict_button, output]))
except ImportError:
    print('ipywidgets is unavailable; enter values below.')
    city_input = input(f"City ({', '.join(available_cities)}): ")
    date_input = input('Date (YYYY-MM-DD): ')
    print(f'Predicted AQI: {make_prediction(city_input, date_input):.1f}')

In [9]:
import calendar
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Prepare historical data once; this does not reload the source files or retrain the model.
graph_df = df.copy()
graph_df['Date'] = pd.to_datetime(graph_df['Date'], errors='coerce')
graph_df['AQI'] = pd.to_numeric(graph_df['AQI'], errors='coerce')
graph_df = graph_df.dropna(subset=['City', 'Date', 'AQI']).copy()
graph_df['City'] = graph_df['City'].astype(str)
graph_df['Year'] = graph_df['Date'].dt.year
graph_df['Month'] = graph_df['Date'].dt.month
graph_cities = sorted(graph_df['City'].unique().tolist())
all_years = sorted(graph_df.loc[graph_df['Year'] >= 2015, 'Year'].unique().tolist())
pollutant_candidates = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
available_pollutants = [c for c in pollutant_candidates if c in graph_df.columns]

city_dropdown = widgets.Dropdown(
    options=graph_cities,
    value='Chandigarh' if 'Chandigarh' in graph_cities else (graph_cities[0] if graph_cities else None),
    description='City:', layout=widgets.Layout(width='300px'), style={'description_width': 'initial'}
)
year_dropdown = widgets.Dropdown(
    options=all_years if all_years else [2023],
    value=max(all_years) if all_years else 2023,
    description='Monthly graph year:', layout=widgets.Layout(width='300px'), style={'description_width': 'initial'}
)
pollutant_dropdown = widgets.Dropdown(
    options=available_pollutants,
    value='PM2.5' if 'PM2.5' in available_pollutants else (available_pollutants[0] if available_pollutants else None),
    description='Pollutant:', layout=widgets.Layout(width='300px'), style={'description_width': 'initial'}
)
CHART_TEMPLATE = 'plotly_white'
PALETTE = px.colors.qualitative.Bold
display(widgets.VBox([
    widgets.HTML('<h3 style="color:#6D28D9">🌍 Interactive AQI Explorer</h3><p>Choose settings, then run individual graph cells below.</p>'),
    widgets.HBox([city_dropdown, year_dropdown]),
    widgets.HBox([pollutant_dropdown])
]))

def selected_city_data():
    """Return a sorted copy of the currently selected city's data."""
    if city_dropdown.value is None:
        print('No city is available in the dataset.')
        return pd.DataFrame()
    data = graph_df.loc[graph_df['City'] == city_dropdown.value].copy().sort_values('Date')
    if data.empty:
        print('No historical records found for this city.')
        return data
    data['Year'] = data['Date'].dt.year
    data['Month'] = data['Date'].dt.month
    data['MonthName'] = data['Date'].dt.month.map(lambda m: calendar.month_abbr[m])
    data['Day_Type'] = data['Date'].dt.dayofweek.map(lambda d: 'Weekend' if d >= 5 else 'Weekday')
    data['Season'] = data['Month'].map({12:'Winter', 1:'Winter', 2:'Spring', 3:'Spring', 4:'Summer', 5:'Summer', 6:'Summer', 7:'Summer', 8:'Monsoon', 9:'Monsoon', 10:'Autumn', 11:'Autumn'})
    return data


In [10]:
city_data = selected_city_data()
if not city_data.empty:
    city_name = city_dropdown.value
    daily = city_data.groupby('Date', as_index=False)['AQI'].mean().sort_values('Date')
    daily['30-day rolling average'] = daily.set_index('Date')['AQI'].rolling('30D', min_periods=1).mean().to_numpy()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=daily['Date'], y=daily['AQI'], mode='lines', name='Daily AQI', line=dict(color='#4F46E5', width=1.5), opacity=0.65, hovertemplate='%{x|%d %b %Y}<br>AQI: %{y:.1f}<extra></extra>'))
    fig.add_trace(go.Scatter(x=daily['Date'], y=daily['30-day rolling average'], mode='lines', name='30-day rolling average', line=dict(color='#F97316', width=3), hovertemplate='%{x|%d %b %Y}<br>30-day average: %{y:.1f}<extra></extra>'))
    fig.update_layout(title=f'🌈 {city_name} — Daily AQI trend (all available dates)', xaxis_title='Date', yaxis_title='AQI', template=CHART_TEMPLATE, hovermode='x unified', legend_title='Series')
    fig.show()
